# ML용 농장현황 전처리

`감염농장_전처리.ipynb`를 먼저 같은 `PROVINCE`로 실행해서 `data/{PROVINCE}/농장현황/`, `data/{PROVINCE}/감염농장/`이 만들어져 있어야 한다 (이 노트북은 그 결과를 그대로 사용).

감염농장의 발생일(`OCCRRNC_DE`) 기준 앞뒤 `WINDOW`(`6months`=6개월 또는 `1year`=1년)에 해당하는 농장현황만 모은다. 연도 단위로 자르면 1월 발생 건이 전년도 후반기 농장현황의 영향을 못 받게 되므로, 발생일 기준으로 정확히 ±WINDOW 윈도우를 잡아 연도 경계를 넘어가도 챙긴다.

**`조사날짜`는 원본 농장현황의 조사날짜를 그대로 유지한다.** ML 학습의 기준 시점(base date)을 감염날짜에 맞추기 위해, 별도로 **`감염날짜`** 컬럼을 새로 만들어 채운다: 농장현황 좌표 반경 `RADIUS_KM`(10km, 시군 경계와 무관) 이내 `최종감염농장/전국_최종감염농장.csv`의 `outbreak_date` 중, 원래 조사날짜와 시간상 가장 가까운 것을 찾아 그 값(연-월-일까지)을 채운다 (시간 거리가 같은 후보가 여러 개면 공간 거리(km)가 더 가까운 쪽을 우선한다). 반경 안에 감염농장이 하나도 없으면, 그 행을 윈도우에 포함시킨 원인인 그 시군의 `occr_date`(감염 발생일)를 대신 채운다.

감염농장이 여러 건이면 같은 농장이 서로 다른 occr_date 윈도우에 걸쳐 잡혀, 채워지는 감염날짜가 윈도우마다 달라질 수 있다. 시군명·농장명·상세구분·소재지지번주소·위도·경도·사육두수가 같으면 같은 농장인데, 이를 처리하는 두 버전을 같이 만든다.

- **`ML_{PROVINCE}_농장현황_{WINDOW}.csv`**: 같은 농장이 여러 윈도우에 걸쳐 잡혀 감염날짜가 여러 개면, 원래 조사날짜와 가장 가까운 감염날짜 1건만 남긴다.
- **`ML_{PROVINCE}_농장현황_{WINDOW}_alldates.csv`**: 조사날짜·감염날짜가 다르면 버리지 않고 전부 남긴다. 둘 다 완전히 같은 행만 중복으로 보고 제거한다.

**감염농장과 조인하지 않고, 농장현황 데이터만 그대로 저장한다** — 라벨은 없고, `감염날짜`는 ML 학습 시 기준 시점으로 쓸 보조 컬럼이다.

**출력**: `ML/ML_{PROVINCE}_농장현황_{WINDOW}.csv`, `ML/ML_{PROVINCE}_농장현황_{WINDOW}_alldates.csv`, 건수 요약은 각각 `ML/ML_농장현황_건수_{WINDOW}.csv` / `ML/ML_농장현황_건수_{WINDOW}_alldates.csv`에 시/도별로 누적 기록


## 1. 시/도·윈도우 선택 (이 두 변수만 바꿔서 재실행)

In [1]:
PROVINCE = "충청북도"
WINDOW = "1year"

## 2. 경로 설정

In [2]:
import glob
import os

import numpy as np
import unicodedata
import pandas as pd

PROVINCE = unicodedata.normalize("NFC", PROVINCE)  # macOS NFD/NFC 정규화 (감염농장_전처리.ipynb와 동일한 이유)

# 이 노트북은 ML/ 폴더에 있고, Jupyter/nbconvert 모두 노트북 파일이 있는 폴더를 작업 디렉터리로 잡으므로
# repo 루트의 data/는 한 단계 위(..)에서 찾는다.
CENSUS_DIR = os.path.join("..", "data", PROVINCE, "농장현황")
INFECTION_DIR = os.path.join("..", "data", PROVINCE, "감염농장")

assert os.path.isdir(CENSUS_DIR) and os.path.isdir(INFECTION_DIR), (
    f"{CENSUS_DIR} 또는 {INFECTION_DIR}가 없습니다. "
    f"먼저 감염농장_전처리.ipynb를 PROVINCE='{PROVINCE}'로 실행하세요."
)
print("농장현황 폴더:", CENSUS_DIR)
print("감염농장 폴더:", INFECTION_DIR)

농장현황 폴더: ../data/충청북도/농장현황
감염농장 폴더: ../data/충청북도/감염농장


## 3. 좌표 결측치 보완 + `전국_최종감염농장.csv` 기준 반경 내 가장 가까운 outbreak_date 찾기

경기도를 제외한 시/도는 원본 농장현황에 `WGS84위도`/`WGS84경도`가 없다. 반경 매칭이 좌표에 의존하므로, 미리 기존에 지오코딩해둔 캐시(`scripts/geocode_cache.json`, 주소를 읍/면/동·리 단위로 잘라 Kakao로 조회해둔 결과)에서 같은 동/리의 좌표를 찾아 결측치를 채운다. 캐시에 없는 주소는 그대로 비워두고(반경 매칭 실패 → occr_date로 대체), 새로 Kakao API를 호출하지는 않는다.

그다음 `감염날짜` 컬럼을 채우기 위해, 농장현황 좌표 반경 `RADIUS_KM`(10km) 이내 감염농장들의 `outbreak_date` 중 원래 조사날짜와 가장 가까운 것(그 날짜, 그리고 그때까지의 시간 거리)을 미리 계산할 함수를 준비한다. 시간 거리가 같은 후보가 여러 개면 공간 거리(km)가 더 가까운 쪽을 우선한다. 반경 안에 감염농장이 하나도 없으면 이 단계에서는 매칭 없음(`NaT`/`+inf`)으로 두고, 다음 단계(윈도우 루프)에서 그 시군의 occr_date로 대체한다.


In [3]:
import json as _json
import re as _re

GEOCODE_CACHE_PATH = os.path.join("..", "scripts", "geocode_cache.json")
with open(GEOCODE_CACHE_PATH, encoding="utf-8") as f:
    _GEOCODE_CACHE = _json.load(f)


def _truncate_to_dong(address):
    """번지/필지/마스킹(***) 등 지번 상세부를 잘라내고 읍/면/동·리까지만 남긴다.
    scripts/geocode_lib.py의 동일 함수와 같은 규칙이어야 캐시 키가 맞는다."""
    tokens = address.split()
    out = []
    for t in tokens:
        if _re.search(r"[0-9*]", t) or t == "외":
            break
        out.append(t)
    return " ".join(out)


def fill_missing_coords(df):
    """WGS84위도/경도가 비어있는 행을, 주소를 동/리 단위로 잘라 기존 지오코딩 캐시에서 찾아 채운다.
    (경기도 제외 시/도는 원본 농장현황에 좌표가 없어서 반경 매칭이 항상 실패하기 때문.)
    캐시에 없는 주소는 그대로 비워두고, 새로 Kakao API를 호출하지는 않는다 (반경 매칭 실패 시
    이후 단계에서 occr_date로 대체됨)."""
    missing = df["WGS84위도"].isna() | df["WGS84경도"].isna()
    if not missing.any():
        return df
    df = df.copy()
    for idx in df.index[missing]:
        addr = df.at[idx, "소재지지번주소"]
        if not isinstance(addr, str):
            continue
        hit = _GEOCODE_CACHE.get(_truncate_to_dong(addr))
        if hit and hit.get("lat") is not None:
            df.at[idx, "WGS84위도"] = hit["lat"]
            df.at[idx, "WGS84경도"] = hit["lon"]
    return df


RADIUS_KM = 10.0
EARTH_RADIUS_KM = 6371.0

INFECTION_NATIONAL_PATH = os.path.join("..", "최종감염농장", "전국_최종감염농장.csv")
infection_national_df = pd.read_csv(INFECTION_NATIONAL_PATH, encoding="utf-8-sig")
infection_national_df["outbreak_date"] = pd.to_datetime(
    infection_national_df["outbreak_date"], format="%Y%m%d", errors="coerce"
)
infection_national_df = infection_national_df.dropna(subset=["outbreak_date", "latitude", "longitude"])

INF_LAT_RAD = np.radians(infection_national_df["latitude"].to_numpy())
INF_LON_RAD = np.radians(infection_national_df["longitude"].to_numpy())
INF_OUTBREAK_DATES = infection_national_df["outbreak_date"].to_numpy("datetime64[ns]")
INF_DATE_DAYS = (INF_OUTBREAK_DATES - np.datetime64("1970-01-01")) / np.timedelta64(1, "D")

# 시간 거리가 동률일 때 공간 거리(km)로 2차 타이브레이크하기 위한 가중치.
# RADIUS_KM(최대 10km)보다 훨씬 큰 값을 곱해서, 시간 거리가 1일만 달라도 항상 그쪽을 우선하게 한다.
TIME_DIST_WEIGHT = 1000.0


def nearest_outbreak_within_radius(df):
    """반경 RADIUS_KM(10km) 이내 감염농장들의 outbreak_date 중, 원래 조사날짜와 가장 가까운 것을
    찾아 (그 outbreak_date, 그때까지의 시간 거리(일 단위))를 행마다 반환한다. 시간 거리가 동률이면
    공간 거리(km)가 더 가까운 outbreak_date를 우선한다. 반경 안에 감염농장이 하나도 없으면
    (NaT, +inf)를 반환한다 (호출하는 쪽에서 occr_date로 대체)."""
    n = len(df)
    if n == 0 or len(INF_DATE_DAYS) == 0:
        return (
            pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]"),
            pd.Series(np.inf, index=df.index, dtype=float),
        )

    lat1 = np.radians(df["WGS84위도"].to_numpy())[:, None]
    lon1 = np.radians(df["WGS84경도"].to_numpy())[:, None]
    lat2 = INF_LAT_RAD[None, :]
    lon2 = INF_LON_RAD[None, :]
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    dist_km = 2 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0)))
    within_radius = dist_km <= RADIUS_KM  # (행 수, 전국 감염농장 수)

    survey_days = (
        df["조사날짜"].to_numpy("datetime64[ns]") - np.datetime64("1970-01-01")
    ) / np.timedelta64(1, "D")
    time_diff_days = np.abs(survey_days[:, None] - INF_DATE_DAYS[None, :])
    time_diff_days = np.where(within_radius, time_diff_days, np.inf)

    # 1차: 시간 거리, 2차(동률 시): 공간 거리(km)
    composite = time_diff_days * TIME_DIST_WEIGHT + dist_km

    best_idx = composite.argmin(axis=1)
    best_time_dist = time_diff_days[np.arange(n), best_idx]
    has_match = np.isfinite(best_time_dist)

    assigned = np.where(has_match, INF_OUTBREAK_DATES[best_idx], np.datetime64("NaT"))

    return (
        pd.Series(assigned, index=df.index, dtype="datetime64[ns]"),
        pd.Series(np.where(has_match, best_time_dist, np.inf), index=df.index, dtype=float),
    )

print(f"✓ 반경 {RADIUS_KM}km 기준 감염농장 {len(infection_national_df)}건 로드")
print(f"✓ 좌표 지오코딩 캐시 {len(_GEOCODE_CACHE)}개 주소 로드")


✓ 반경 10.0km 기준 감염농장 1173건 로드
✓ 좌표 지오코딩 캐시 3105개 주소 로드


## 4. 발생일 기준 ±WINDOW 농장현황 윈도우 추출 + 감염날짜 컬럼 채우기 (1건 버전 + alldates 버전 동시 생성)


In [4]:
DEDUP_COLS = ["시군명", "농장명", "상세구분", "소재지지번주소", "WGS84위도", "WGS84경도", "사육두수(마리)"]
ALLDATES_DEDUP_COLS = DEDUP_COLS + ["조사날짜", "감염날짜"]  # alldates 버전은 조사날짜·감염날짜까지 같은 행만 중복으로 본다

WINDOW_OFFSETS = {"6months": pd.DateOffset(months=6), "1year": pd.DateOffset(years=1)}
window_offset = WINDOW_OFFSETS[WINDOW]

sigun_frames = []
sigun_frames_alldates = []

for infection_file in sorted(glob.glob(os.path.join(INFECTION_DIR, "*_감염농장.csv"))):
    sigun = os.path.splitext(os.path.basename(infection_file))[0].replace("_감염농장", "")
    census_file = os.path.join(CENSUS_DIR, f"{sigun}_농장현황.csv")

    if not os.path.exists(census_file):
        print(f"  ⚠️  {sigun}: 농장현황 파일 없음, 스킵")
        continue

    infection_df = pd.read_csv(infection_file, encoding="utf-8-sig")
    census_df = pd.read_csv(census_file, encoding="utf-8-sig")
    census_df["조사날짜"] = pd.to_datetime(census_df["조사날짜"])
    census_df = fill_missing_coords(census_df)

    # 이 시군에서 감염이 발생한 날짜들만 (감염농장이 여러 건이어도 같은 날짜면 한 번만 처리)
    occr_dates = sorted(pd.to_datetime(infection_df["OCCRRNC_DE"], format="%Y%m%d", errors="coerce").dropna().unique())

    for occr_date in occr_dates:
        occr_date = pd.Timestamp(occr_date)
        start = occr_date - window_offset
        end = occr_date + window_offset
        windowed = census_df[(census_df["조사날짜"] >= start) & (census_df["조사날짜"] <= end)].copy()
        before = len(windowed)

        # 감염날짜를 채운다: 반경 10km 이내 가장 가까운 outbreak_date, 없으면 이 윈도우의 occr_date
        assigned_dates, assign_dist = nearest_outbreak_within_radius(windowed)
        no_match = assigned_dates.isna()
        fallback_dist = (windowed["조사날짜"] - occr_date).abs().dt.days.astype(float)
        assigned_dates = assigned_dates.where(~no_match, occr_date)
        assign_dist = assign_dist.where(~no_match, fallback_dist)
        windowed = windowed.assign(감염날짜=assigned_dates, _assign_dist=assign_dist)

        windowed_picked = (
            windowed.sort_values("_assign_dist", kind="stable")
            .drop_duplicates(subset=DEDUP_COLS, keep="first")
            .sort_index()
        )
        windowed_alldates = windowed.drop_duplicates(subset=ALLDATES_DEDUP_COLS, keep="first")

        sigun_frames.append(windowed_picked)
        sigun_frames_alldates.append(windowed_alldates)
        print(
            f"  {sigun} {occr_date.date()} ±{WINDOW} ({start.date()}~{end.date()}): "
            f"{before} → 1건 버전 {len(windowed_picked)}건 / alldates {len(windowed_alldates)}건 "
            f"(반경 밖 occr_date 대체 {int(no_match.sum())}건)"
        )

census_window_df = pd.concat(sigun_frames, ignore_index=True) if sigun_frames else pd.DataFrame()
census_window_alldates_df = pd.concat(sigun_frames_alldates, ignore_index=True) if sigun_frames_alldates else pd.DataFrame()
print(f"\n✓ 윈도우 합계: 1건 버전 {len(census_window_df)}건 / alldates {len(census_window_alldates_df)}건")


  ⚠️  괴산군: 농장현황 파일 없음, 스킵
  ⚠️  영동군: 농장현황 파일 없음, 스킵
  ⚠️  옥천군: 농장현황 파일 없음, 스킵
  음성군 2003-12-12 ±1year (2002-12-12~2004-12-12): 0 → 1건 버전 0건 / alldates 0건 (반경 밖 occr_date 대체 0건)
  음성군 2003-12-17 ±1year (2002-12-17~2004-12-17): 0 → 1건 버전 0건 / alldates 0건 (반경 밖 occr_date 대체 0건)
  음성군 2003-12-19 ±1year (2002-12-19~2004-12-19): 0 → 1건 버전 0건 / alldates 0건 (반경 밖 occr_date 대체 0건)
  음성군 2003-12-24 ±1year (2002-12-24~2004-12-24): 0 → 1건 버전 0건 / alldates 0건 (반경 밖 occr_date 대체 0건)
  음성군 2014-02-07 ±1year (2013-02-07~2015-02-07): 0 → 1건 버전 0건 / alldates 0건 (반경 밖 occr_date 대체 0건)
  음성군 2014-02-10 ±1year (2013-02-10~2015-02-10): 0 → 1건 버전 0건 / alldates 0건 (반경 밖 occr_date 대체 0건)
  음성군 2014-02-19 ±1year (2013-02-19~2015-02-19): 0 → 1건 버전 0건 / alldates 0건 (반경 밖 occr_date 대체 0건)
  음성군 2014-02-20 ±1year (2013-02-20~2015-02-20): 0 → 1건 버전 0건 / alldates 0건 (반경 밖 occr_date 대체 0건)
  음성군 2014-02-21 ±1year (2013-02-21~2015-02-21): 0 → 1건 버전 0건 / alldates 0건 (반경 밖 occr_date 대체 0건)
  음성군 2014-02-23 ±1year (2013-0

  진천군 2016-12-10 ±1year (2015-12-10~2017-12-10): 0 → 1건 버전 0건 / alldates 0건 (반경 밖 occr_date 대체 0건)
  진천군 2016-12-14 ±1year (2015-12-14~2017-12-14): 0 → 1건 버전 0건 / alldates 0건 (반경 밖 occr_date 대체 0건)
  진천군 2016-12-21 ±1year (2015-12-21~2017-12-21): 0 → 1건 버전 0건 / alldates 0건 (반경 밖 occr_date 대체 0건)
  진천군 2022-01-22 ±1year (2021-01-22~2023-01-22): 0 → 1건 버전 0건 / alldates 0건 (반경 밖 occr_date 대체 0건)
  진천군 2022-02-02 ±1year (2021-02-02~2023-02-02): 0 → 1건 버전 0건 / alldates 0건 (반경 밖 occr_date 대체 0건)
  진천군 2022-02-08 ±1year (2021-02-08~2023-02-08): 0 → 1건 버전 0건 / alldates 0건 (반경 밖 occr_date 대체 0건)
  진천군 2022-02-09 ±1year (2021-02-09~2023-02-09): 0 → 1건 버전 0건 / alldates 0건 (반경 밖 occr_date 대체 0건)
  진천군 2022-02-14 ±1year (2021-02-14~2023-02-14): 0 → 1건 버전 0건 / alldates 0건 (반경 밖 occr_date 대체 0건)
  진천군 2022-10-27 ±1year (2021-10-27~2023-10-27): 0 → 1건 버전 0건 / alldates 0건 (반경 밖 occr_date 대체 0건)
  진천군 2024-12-28 ±1year (2023-12-28~2025-12-28): 0 → 1건 버전 0건 / alldates 0건 (반경 밖 occr_date 대체 0건)
  진천군 2025

  진천군 2025-12-30 ±1year (2024-12-30~2026-12-30): 95 → 1건 버전 95건 / alldates 95건 (반경 밖 occr_date 대체 54건)
  ⚠️  청원군: 농장현황 파일 없음, 스킵
  ⚠️  청주시: 농장현황 파일 없음, 스킵
  ⚠️  충주시: 농장현황 파일 없음, 스킵



✓ 윈도우 합계: 1건 버전 1181건 / alldates 1181건


## 5. 시군 간 최종 중복 제거 후 저장 → `ML/ML_{PROVINCE}_농장현황_{WINDOW}.csv` (+ `_alldates`), 건수 요약 누적

같은 농장이 서로 다른 occr_date 윈도우에 걸쳐 잡히면, 감염날짜(반경 매칭 또는 대체된 occr_date)가 윈도우마다 다를 수 있다. 1건 버전은 원래 조사날짜와 가장 가까운 감염날짜 1건만 남기고, alldates 버전은 조사날짜·감염날짜가 다르면 모두 남긴다.


In [5]:
before_total = len(census_window_df)
final_df = (
    census_window_df.sort_values("_assign_dist", kind="stable")
    .drop_duplicates(subset=DEDUP_COLS, keep="first")
    .reset_index(drop=True)
)

before_total_alldates = len(census_window_alldates_df)
final_alldates_df = census_window_alldates_df.drop_duplicates(subset=ALLDATES_DEDUP_COLS, keep="first").reset_index(drop=True)

print(f"1건 버전: {before_total}건 → {len(final_df)}건")
print(f"alldates: {before_total_alldates}건 → {len(final_alldates_df)}건")


1건 버전: 1181건 → 261건
alldates: 1181건 → 643건


## 6. 닭/오리 세부분류 정규화 + 컬럼 순서 통일

시/도마다 `축종명`에 세부 항목(예: `육계`, `종계`, `육용오리`)이 그대로 들어있거나 `상세구분`에 들어있는 등 형식이 다르다. `livestock_codes.csv`(정부 API의 `LVSTCKSPC_NM`에서 파생된 표준 코드표) 기준으로, 닭/오리 관련 단어가 `축종명`이나 `상세구분`에 있으면 `축종명`을 닭/오리로, `상세구분`을 표준 세부값으로 통일한다.

- `종계/산란계`처럼 한 행에 표준 세부값이 두 개 들어있으면 둘 다 보존해서 `/`로 합친다 (임의로 하나를 버리지 않음).
- 닭/오리 둘 다 섞였거나 닭·오리가 아닌 다른 축종(꿩·타조·산양 등)과 섞인 행은 원본 데이터 자체의 모순(예: 꿩 농장인데 `상세구분`에 산란계가 적힌 경우)이라 임의로 고치지 않고 원본 값 그대로 둔다.
- 컬럼 순서를 `시군명, 농장명, 축종명, 상세구분, 사육두수(마리), 소재지지번주소, WGS84위도, WGS84경도, 조사날짜, 감염날짜` 10개로 통일한다 (경기도에만 있는 소재지우편번호/소재지도로명주소/데이터기준일자/비고는 제외).


In [6]:
import re

STANDARD_COLS = ["시군명", "농장명", "축종명", "상세구분", "사육두수(마리)", "소재지지번주소", "WGS84위도", "WGS84경도", "조사날짜", "감염날짜"]

# livestock_codes.csv의 닭/오리 상세구분 어휘 기준, 구체적인 단어를 먼저 검사한다 (예: "산란중추"를 "산란계"보다 먼저 검사)
CHICKEN_DETAIL_PATTERNS = [
    ("토종닭종계", "종계"),
    ("육용종계", "육용종계"),
    ("산란종계", "산란종계"),
    ("산란중추", "산란중추"),
    ("산란육성계", "산란중추"),
    ("산란계", "산란계"),
    ("원종계", "원종계"),
    ("종계", "종계"),
    ("토종닭", "토종닭"),
    ("오골계", "토종닭"),
    ("백세미", "백세미"),
    ("삼계", "육계"),
    ("육계", "육계"),
    ("육닭", "육계"),
    ("관상계", "기타"),
    ("일괄", "일괄"),
    ("비분류", "비분류"),
]
DUCK_DETAIL_PATTERNS = [
    ("종오리", "종오리"),
    ("육용오리", "육용오리"),
    ("산란오리", "산란오리"),
    ("페킹덕", "페킹덕"),
]


def classify_token(tok):
    tok = re.sub(r"\([^)]*\)", "", tok).strip()  # "닭(부화업)" 같은 괄호 설명 제거
    if not tok:
        return None
    for pat, detail in DUCK_DETAIL_PATTERNS:
        if pat in tok:
            return ("오리", detail)
    if "오리" in tok:
        return ("오리", None)
    for pat, detail in CHICKEN_DETAIL_PATTERNS:
        if pat in tok:
            return ("닭", detail)
    if "닭" in tok:
        return ("닭", None)
    return ("기타", tok)


def classify_poultry(axis_raw, detail_raw):
    """축종명/상세구분 원본 값을 받아 (새 축종명, 새 상세구분)을 반환한다.
    닭/오리가 아닌 축종이거나, 닭·오리가 섞이거나 다른 축종과 섞인 모호한 행은 원본 그대로 반환한다."""
    parts = []
    if isinstance(axis_raw, str):
        parts.append(axis_raw)
    if isinstance(detail_raw, str) and detail_raw != axis_raw:
        parts.append(detail_raw)
    if not parts:
        return (axis_raw, detail_raw)

    tokens = [t.strip() for p in parts for t in re.split(r"[,/+]", p) if t.strip()]

    species_set, chicken_details, duck_details, has_other = set(), [], [], False
    for tok in tokens:
        c = classify_token(tok)
        if c is None:
            continue
        sp, detail = c
        species_set.add(sp)
        if sp == "닭" and detail and detail not in chicken_details:
            chicken_details.append(detail)
        if sp == "오리" and detail and detail not in duck_details:
            duck_details.append(detail)
        if sp == "기타":
            has_other = True

    poultry_species = species_set & {"닭", "오리"}
    if not poultry_species or len(poultry_species) > 1 or has_other:
        return (axis_raw, detail_raw)  # 닭/오리 무관 또는 모호 -> 원본 유지

    if "닭" in poultry_species:
        if "종계" in chicken_details and ("육용종계" in chicken_details or "산란종계" in chicken_details):
            chicken_details.remove("종계")  # 육용종계/산란종계가 더 구체적이므로 일반 종계는 버림
        return ("닭", "/".join(chicken_details) if chicken_details else pd.NA)
    return ("오리", "/".join(duck_details) if duck_details else pd.NA)


def normalize_poultry_df(df):
    df = df.copy()
    if "축종명" not in df.columns:
        df["축종명"] = pd.NA
    if "상세구분" not in df.columns:
        df["상세구분"] = pd.NA
    normalized = [classify_poultry(a, d) for a, d in zip(df["축종명"], df["상세구분"])]
    df["축종명"] = [n[0] for n in normalized]
    df["상세구분"] = [n[1] for n in normalized]
    return df[STANDARD_COLS]


final_df = normalize_poultry_df(final_df)
final_alldates_df = normalize_poultry_df(final_alldates_df)

# 정규화 전에는 상세구분이 서로 달라(예: NaN vs "오리") 구분 제거를 통과했던 행이,
# 정규화 후 같은 값(예: 둘 다 NaN)으로 합쳐지면서 완전한 중복 행이 될 수 있다
# (원본 농장현황 자체에 똑같은 농장/조사날짜가 두 번 들어있는 경우, 주로 경기도).
# 정규화된 컬럼 기준으로 한 번 더 완전 중복만 제거한다.
before_norm_dedup = len(final_df)
final_df = final_df.drop_duplicates(subset=STANDARD_COLS, keep="first").reset_index(drop=True)
before_norm_dedup_alldates = len(final_alldates_df)
final_alldates_df = final_alldates_df.drop_duplicates(subset=STANDARD_COLS, keep="first").reset_index(drop=True)

print("✓ 닭/오리 정규화 + 컬럼 순서 통일 완료 (1건 버전)")
print(f"  정규화 후 완전 중복 제거: {before_norm_dedup}건 → {len(final_df)}건")
print("축종명 분포:", final_df["축종명"].value_counts(dropna=False).to_dict())
print("✓ 닭/오리 정규화 + 컬럼 순서 통일 완료 (alldates)")
print(f"  정규화 후 완전 중복 제거: {before_norm_dedup_alldates}건 → {len(final_alldates_df)}건")
print("축종명 분포:", final_alldates_df["축종명"].value_counts(dropna=False).to_dict())

✓ 닭/오리 정규화 + 컬럼 순서 통일 완료 (1건 버전)
  정규화 후 완전 중복 제거: 261건 → 261건
축종명 분포: {'닭': 171, '오리': 79, '메추리': 6, '타조': 1, '면양, 타조, 염소': 1, '육계, 산양': 1, '꿩': 1, '한우, 메추리': 1}
✓ 닭/오리 정규화 + 컬럼 순서 통일 완료 (alldates)
  정규화 후 완전 중복 제거: 643건 → 643건
축종명 분포: {'닭': 461, '오리': 156, '메추리': 16, '면양, 타조, 염소': 3, '육계, 산양': 3, '꿩': 2, '타조': 1, '한우, 메추리': 1}


## 7. 저장 → `ML/ML_{PROVINCE}_농장현황_{WINDOW}.csv` (+ `_alldates`), 건수 요약 누적

In [7]:
def save_with_count(df, suffix):
    out_path = f"ML_{PROVINCE}_농장현황_{suffix}.csv"  # 노트북이 이미 ML/ 안에 있으므로 그대로 저장
    df.to_csv(out_path, index=False, encoding="utf-8")

    count_path = f"ML_농장현황_건수_{suffix}.csv"
    if os.path.exists(count_path):
        count_df = pd.read_csv(count_path, encoding="utf-8-sig")
        count_df = count_df[count_df["시도"] != PROVINCE]
    else:
        count_df = pd.DataFrame(columns=["시도", "건수"])

    new_row = pd.DataFrame([{"시도": PROVINCE, "건수": len(df)}])
    count_df = pd.concat([count_df, new_row], ignore_index=True)
    count_df.to_csv(count_path, index=False, encoding="utf-8")

    print(f"✓ 저장 완료: {out_path} ({len(df)}건)")
    print(f"✓ 건수 요약 갱신: {count_path}")
    print(count_df.to_string(index=False))
    return count_path


save_with_count(final_df, WINDOW)
print()
save_with_count(final_alldates_df, f"{WINDOW}_alldates")

✓ 저장 완료: ML_충청북도_농장현황_1year.csv (261건)
✓ 건수 요약 갱신: ML_농장현황_건수_1year.csv
  시도   건수
 경기도 3662
전라남도  832
전라북도  847
충청남도  226
충청북도  261

✓ 저장 완료: ML_충청북도_농장현황_1year_alldates.csv (643건)
✓ 건수 요약 갱신: ML_농장현황_건수_1year_alldates.csv
  시도   건수
 경기도 4187
전라남도  880
전라북도 2588
충청남도  730
충청북도  643


'ML_농장현황_건수_1year_alldates.csv'